In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import pandas as pd
import nibabel as nib
import numpy as np
import re # Import regex module

# Define the base directory for your data on Google Drive
base_drive_path = '/content/drive/MyDrive/Pipeline/first_run/Volumes' # Corrected path

# List of persons to process
persons = ['Person1', 'Person2']

def calculate_nifti_volume(filepath):
    """
    Calculates the 3D volume of a NIfTI file based on voxel dimensions and non-zero voxels.
    Assumes non-zero voxels represent the structure of interest.
    """
    try:
        img = nib.load(filepath)
        header = img.header
        voxel_dims = header.get_zooms() # Get voxel dimensions (dx, dy, dz)
        # Count non-zero voxels (assuming 0 is background and other values are foreground)
        num_voxels = np.sum(img.get_fdata() > 0)
        volume_mm3 = num_voxels * voxel_dims[0] * voxel_dims[1] * voxel_dims[2]
        return volume_mm3
    except Exception as e:
        print(f"Error calculating volume for {filepath}: {e}")
        return None

# Dictionary to store results for each person, organized by tooth
all_volumes = {person: {} for person in persons}

In [ ]:
for person in persons:
    person_path = os.path.join(base_drive_path, person)
    if not os.path.exists(person_path):
        print(f"Warning: Path not found for {person}: {person_path}")
        continue

    print(f"Processing {person} in {person_path}...")

    # Walk through the person's directory to find NIfTI files
    for root, _, files in os.walk(person_path):
        for file in files:
            if file.endswith(('.nii', '.nii.gz')):
                print(f"  - Calculating volume for: {file}") # Added progress indicator
                filepath = os.path.join(root, file)
                volume = calculate_nifti_volume(filepath)

                if volume is not None:
                    file_lower = file.lower()
                    tooth_fdi = None
                    structure_type = None

                    pulp_match = None
                    tooth_match = None

                    # Try to match pulp first: e.g., 'upper_right_central_incisor_pulp_fdi111.nii.gz' for FDI 11
                    # Pattern: 'pulp_fdi1' followed by two digits for the FDI number
                    pulp_match = re.search(r'pulp_fdi1(\d{2})', file_lower)
                    if pulp_match:
                        structure_type = 'Pulp_Volume_mm3'
                        tooth_fdi = pulp_match.group(1)
                    else:
                        # If not pulp, try to match tooth: e.g., 'upper_right_central_incisor_fdi11.nii.gz' for FDI 11
                        # Pattern: 'fdi' followed by two digits for the FDI number
                        tooth_match = re.search(r'fdi(\d{2})', file_lower)
                        if tooth_match:
                            structure_type = 'Tooth_Volume_mm3'
                            tooth_fdi = tooth_match.group(1)

                    if tooth_fdi and structure_type:
                        if tooth_fdi not in all_volumes[person]:
                            all_volumes[person][tooth_fdi] = {}
                        all_volumes[person][tooth_fdi][structure_type] = volume
                    else:
                        # Only print warning if parsing failed for files expected to be tooth/pulp
                        # (i.e., files that contain 'fdi' or 'pulp' but didn't match the specific patterns)
                        if ('fdi' in file_lower or 'pulp' in file_lower or 'răng' in file_lower or 'tuỷ_răng' in file_lower) and not (pulp_match or tooth_match):
                            print(f"    -> Warning: Could not parse unexpected pattern: {file}")
                        # Files like 'mandible.nii.gz' (without fdi/pulp keywords) will now silently be ignored as intended.

print("Finished volume calculation.")

## Step 4: Create DataFrame

This final step converts the collected volume data into a pandas DataFrame, with columns for each tooth's volume and pulp volume, and rows representing each person.

In [ ]:
# Prepare data for DataFrame construction
flat_data = []
for person, teeth_data in all_volumes.items():
    row = {'Person': person}
    for tooth_fdi, volumes in teeth_data.items():
        tooth_vol = volumes.get('Tooth_Volume_mm3')
        pulp_vol = volumes.get('Pulp_Volume_mm3')

        if tooth_vol is not None:
            row[f'Tooth_{tooth_fdi}_Volume_mm3'] = tooth_vol
        if pulp_vol is not None:
            row[f'Pulp_{tooth_fdi}_Volume_mm3'] = pulp_vol

        if tooth_vol is not None and pulp_vol is not None and tooth_vol != 0:
            row[f'Pulp_Tooth_Ratio_{tooth_fdi}'] = pulp_vol / tooth_vol
        elif tooth_vol is not None and pulp_vol is not None and tooth_vol == 0:
            row[f'Pulp_Tooth_Ratio_{tooth_fdi}'] = np.nan # Handle division by zero
    flat_data.append(row)

# Convert the prepared data into a DataFrame
df_volumes = pd.DataFrame(flat_data)

# Set 'Person' as the index for better readability
df_volumes.set_index('Person', inplace=True)

# Display the resulting DataFrame
print("Generated DataFrame:")
display(df_volumes)

# Fill any missing values (e.g., if a tooth/pulp was not found for a person) with NaN or 0
# df_volumes = df_volumes.fillna(0) # Use 0 if missing volumes should be treated as such

In [ ]:
# Lưu DataFrame thành tệp Excel trên Google Drive
output_path = '/content/drive/MyDrive/Pipeline/first_run/volumes_report.xlsx'
df_volumes.to_excel(output_path)
print(f"Dữ liệu đã được lưu thành công tại: {output_path}")